In [24]:
import pandas as pd
from transformers import pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load dataset
df = pd.read_csv("dataset.csv", encoding="utf-8-sig")

# Validate required columns
required_columns = {"id", "message", "label", "split"}
missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

df["label"] = df["label"].astype(int)

print("Dataset loaded successfully!")
print("Total messages:", len(df))
print("\nDataset split:")
print(df["split"].value_counts())

display(df.head())

Dataset loaded successfully!
Total messages: 60

Dataset split:
split
train    40
test     20
Name: count, dtype: int64


,id,message,label,intent_type,case_type,group_id,split
0,M01,"Could you quote a second website? Also, the we...",0,support,mixed,G01,test
1,M02,The updated homepage for the website you're de...,0,support,straightforward,G08,train
2,M03,How much would you charge to build a five-page...,1,sales,straightforward,G23,train
3,M04,Our invoice for the website redesign includes ...,0,support,straightforward,G02,test
4,M05,How much would it cost to add a French-languag...,1,sales,paraphrase,G46,train


In [25]:
# Use only the test split for final evaluation
test_df = df[df["split"] == "test"].copy()

test_messages = test_df["message"].tolist()
true_labels = test_df["label"].tolist()

print("Test messages:", len(test_messages))
print("Support messages:", true_labels.count(0))
print("Sales messages:", true_labels.count(1))

display(test_df[["id", "message", "label"]].head())

Test messages: 20
Support messages: 10
Sales messages: 10


,id,message,label
0,M01,"Could you quote a second website? Also, the we...",0
3,M04,Our invoice for the website redesign includes ...,0
5,M06,What features would you include if you built l...,1
8,M09,"Before we commission development, can you expl...",1
9,M10,Can you build a WordPress website for a compan...,1


In [26]:
MODEL_NAME = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"

classifier = pipeline("zero-shot-classification", model=MODEL_NAME)

candidate_labels = [
    "an existing customer service problem, complaint, billing issue, technical issue, or follow-up on an existing service request",
    "a new sales inquiry, purchase, subscription, service plan, product order, or request for a price quotation",
]

print("Selected model loaded successfully!")
print("Model:", MODEL_NAME)
print("Routing categories defined!")

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 986.95it/s]


Selected model loaded successfully!
Model: MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
Routing categories defined!


In [27]:
predicted_labels = []
predicted_flags = []
prediction_scores = []

for message in test_messages:
    result = classifier(message, candidate_labels, multi_label=False)

    label = result["labels"][0]
    score = result["scores"][0]

    if label.startswith("an existing customer service"):
        flag = 0
        display_label = "Support"
    else:
        flag = 1
        display_label = "Sales"

    predicted_labels.append(display_label)
    predicted_flags.append(flag)
    prediction_scores.append(score)

print("Classification completed!")
print("Total predictions:", len(predicted_flags))

Classification completed!
Total predictions: 20


In [28]:
accuracy = accuracy_score(true_labels, predicted_flags)

print("=== FINAL MODEL EVALUATION ===")
print("Model:", MODEL_NAME)
print(f"Accuracy: {accuracy:.1%}")
print(
    f"Correct predictions: "
    f"{sum(t == p for t, p in zip(true_labels, predicted_flags))}"
    f" / {len(true_labels)}"
)

print("\n=== CLASSIFICATION REPORT ===")
print(
    classification_report(
        true_labels,
        predicted_flags,
        labels=[0, 1],
        target_names=["Support", "Sales"],
        zero_division=0,
    )
)

print("=== CONFUSION MATRIX ===")
print("Rows = actual labels; columns = predicted labels")
print("Order: Support (0), Sales (1)")
print(confusion_matrix(true_labels, predicted_flags, labels=[0, 1]))

=== FINAL MODEL EVALUATION ===
Model: MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
Accuracy: 75.0%
Correct predictions: 15 / 20

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

     Support       0.67      1.00      0.80        10
       Sales       1.00      0.50      0.67        10

    accuracy                           0.75        20
   macro avg       0.83      0.75      0.73        20
weighted avg       0.83      0.75      0.73        20

=== CONFUSION MATRIX ===
Rows = actual labels; columns = predicted labels
Order: Support (0), Sales (1)
[[10  0]
 [ 5  5]]


In [29]:
results_df = test_df[["id", "message", "label"]].copy()

results_df["predicted_label"] = predicted_labels
results_df["predicted_flag"] = predicted_flags
results_df["score"] = prediction_scores

results_df["correct"] = results_df["label"] == results_df["predicted_flag"]

print("=== INCORRECT PREDICTIONS ===")
errors_df = results_df[~results_df["correct"]]

if errors_df.empty:
    print("No incorrect predictions.")
else:
    display(errors_df.reset_index(drop=True))

# Save all test predictions
results_df.to_csv("final_results.csv", index=False, encoding="utf-8-sig")

print("\nResults saved to final_results.csv")

=== INCORRECT PREDICTIONS ===


,id,message,label,predicted_label,predicted_flag,score,correct
0,M06,What features would you include if you built l...,1,Support,0,0.790449,False
1,M09,"Before we commission development, can you expl...",1,Support,0,0.876105,False
2,M10,Can you build a WordPress website for a compan...,1,Support,0,0.704707,False
3,M14,I'd like to schedule a consultation about buil...,1,Support,0,0.922761,False
4,M24,What pages and features are included in your b...,1,Support,0,0.618377,False



Results saved to final_results.csv



## Decision Model — Configuration & Results

### Model Configuration
- Model: MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
- Model Type: Pretrained Zero-shot Classification
- Framework: Hugging Face Transformers
- License: MIT
- Fine-tuning: None

### Classification Method
يستقبل الmodel Customer Message ويقارنها مع Candidate Labels لتحديد التصنيف المناسب.

- Support → 0
- Sales → 1

تم اختبار Candidate Labels بصيغتين:
- Original Labels: Accuracy = 70% (14/20)
- Descriptive Labels: Accuracy = 75% (15/20)

### Evaluation Limitation
تم تعديل Candidate Labels بعد مراجعة نتائج الاختبار الأول، ولذلك تُعتبر نتيجة 75% تقييمًا استكشافيًا (Exploratory).

### Source
https://huggingface.co/MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
